# Preprocesamiento

- Objetivo: Preparar el conjunto de datos para el entrenamiento de modelos de Machine Learning mediante tareas de limpieza, selección y transformación de variables.

## Checklist de Preprocesamiento y Feature Engineering

### 1. Limpieza inicial

* [X] Eliminar variable `ID`.
* [X] Eliminar variable `oral` (varianza nula).
* [X] Revisar tratamiento de registros duplicados.
* [X] Confirmar consistencia entre train y test.

### 2. Clasificación de variables

* [X] Identificar variables categóricas.
* [X] Identificar variables ordinales/discretizadas.
* [X] Identificar variables numéricas continuas.

### 3. Transformación de variables

* [X] Definir encoding para variables categóricas.
* [X] Definir tratamiento para variables ordinales.
* [X] Revisar necesidad de escalado/normalización.

### 4. Selección de variables

* [X] Evaluar eliminación de variables poco informativas.
* [X] Conservar variables con evidencia de asociación con la variable objetivo (`gender`, `hemoglobin`, `dental caries`, `tartar`, entre otras).

### 5. Dataset final para modelado

* [X] Generar conjunto final de entrenamiento.
* [X] Generar conjunto final de prueba.
* [X] Verificar dimensiones finales.


# Importación de Librerias

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# PATHS

In [2]:
BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Verificación PATHS

In [3]:
print(BASE_DIR)
print(RAW_DIR)
print(PROCESSED_DIR)

c:\Users\sdecicco\OneDrive - loteriadelaciudad.gob.ar\Escritorio\Diplomatura en IA\TP2
c:\Users\sdecicco\OneDrive - loteriadelaciudad.gob.ar\Escritorio\Diplomatura en IA\TP2\data\raw
c:\Users\sdecicco\OneDrive - loteriadelaciudad.gob.ar\Escritorio\Diplomatura en IA\TP2\data\processed


# Carga de los datasets

In [4]:
train = pd.read_csv(RAW_DIR / "smoking_prediction.xlsx - smoking_prediction.csv")

test = pd.read_csv(RAW_DIR / "smoking_prediction_entrega.xlsx - smoking_prediction.csv")

# Dimensiones DataSets

In [5]:
print(train.shape)
print(test.shape)

(50000, 27)
(5692, 26)


# Eliminación de variables sin valor predictivo

In [6]:
train_fe = train.copy()
test_fe = test.copy()

train_fe.drop(columns=['ID', 'oral'], inplace=True)
test_fe.drop(columns=['ID', 'oral'], inplace=True)

# Registros duplicados

In [7]:
train_fe.duplicated().sum()

np.int64(5448)

In [8]:
duplicados_pct = (
    train_fe.duplicated().sum()
    / len(train_fe)
) * 100

duplicados_pct

np.float64(10.896)

# Revisión de registros duplicados

- Luego de eliminar las variables `ID` y `oral`, se realizó una nueva revisión de registros duplicados.

- Se detectaron `5448 registros duplicados`, involucrando un total de `10896 observaciones`. Estos resultados coinciden con los obtenidos durante la etapa de Discovery.

- Dado que aún no se determinó si estos registros corresponden a errores de duplicación o a observaciones válidas con características idénticas, se decidió conservarlos temporalmente y documentar el hallazgo para futuras evaluaciones durante el proceso de modelado.


# Clasificación de variables

# Variables categóricas binarias
- gender
- dental caries
- tartar

# Variables discretizadas / ordinales
- eyesight(left)
- eyesight(right)
- Urine protein
- hearing(left)
- hearing(right)

# Variables numéricas
- age
- height(cm)
- weight(kg)
- waist(cm)
- systolic
- relaxation
- fasting blood sugar
- Cholesterol
- triglyceride
- HDL
- LDL
- hemoglobin
- serum creatinine
- AST
- ALT
- Gtp

# Revisión de variables categóricas

In [9]:
train_fe[['gender',
          'dental caries',
          'tartar']].head()

,gender,dental caries,tartar
0,F,0,Y
1,F,0,Y
2,M,0,N
3,M,0,Y
4,F,0,N


In [10]:
train_fe[['dental caries']].describe()

,dental caries
count,50000.000000
mean,0.213120
std,0.409516
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [11]:
train_fe['dental caries'].value_counts()

dental caries
0    39344
1    10656
Name: count, dtype: int64

In [12]:
train_fe['gender'] = train_fe['gender'].map({
    'F': 0,
    'M': 1
})

test_fe['gender'] = test_fe['gender'].map({
    'F': 0,
    'M': 1
})

In [13]:
train_fe['tartar'] = train_fe['tartar'].map({
    'N': 0,
    'Y': 1
})

test_fe['tartar'] = test_fe['tartar'].map({
    'N': 0,
    'Y': 1
})

In [14]:
train_fe[['gender', 'tartar']].head()

,gender,tartar
0,0,1
1,0,1
2,1,0
3,1,1
4,0,0


In [15]:
test_fe[['gender', 'tartar']].head()

,gender,tartar
0,1,1
1,1,0
2,1,1
3,1,0
4,1,1


### Codificación de variables categóricas

- Se identificaron dos variables categóricas con representación textual: `gender` y `tartar`.

- Para permitir su utilización por los algoritmos de Machine Learning, se realizó una codificación binaria o label encoding manual:

* `gender`: F → 0, M → 1
* `tartar`: N → 0, Y → 1

- Las restantes variables ya se encontraban representadas numéricamente y no requirieron transformaciones adicionales en esta etapa.


# Variables codificadas:
- gender
- tartar

# Variables binarias ya codificadas:
- dental caries
- smoking (target)

# Separar features y target

In [16]:
X = train_fe.drop(columns=['smoking'])
y = train_fe['smoking']

# Importar train_test_split

In [17]:
from sklearn.model_selection import train_test_split

# Generar conjuntos de entrenamiento y prueba

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Verificar dimensiones

In [19]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("test_fe:", test_fe.shape)

X_train: (40000, 24)
X_test : (10000, 24)
y_train: (40000,)
y_test : (10000,)
test_fe: (5692, 24)


# Verificar distribución de la target

In [20]:
print(y.value_counts(normalize=True))

print(y_train.value_counts(normalize=True))

print(y_test.value_counts(normalize=True))

smoking
0    0.63342
1    0.36658
Name: proportion, dtype: float64
smoking
0    0.633425
1    0.366575
Name: proportion, dtype: float64
smoking
0    0.6334
1    0.3666
Name: proportion, dtype: float64


- El conjunto de datos fue dividido en entrenamiento (80%) y prueba (20%) utilizando train_test_split. Se aplicó estratificación sobre la variable objetivo (smoking) para conservar la proporción original de fumadores y no fumadores en ambos subconjuntos. La distribución resultante se mantuvo prácticamente idéntica a la observada en el dataset original.

# Generación de datos limpios y procesados listos para modelar

In [21]:
# Guardar datasets procesados

X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
X_test= pd.read_csv(PROCESSED_DIR / "X_test.csv")

y_train= pd.read_csv(PROCESSED_DIR / "y_train.csv")
y_test= pd.read_csv(PROCESSED_DIR / "y_test.csv")

test_fe = pd.read_csv(PROCESSED_DIR / "test_fe.csv")


print("Datasets procesados guardados correctamente.")

Datasets procesados guardados correctamente.
